In [11]:
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path

# Определяем путь к outputs (на уровень выше)
outputs_dir = Path.cwd().parent / "outputs"
outputs_dir.mkdir(exist_ok=True)

print(f"Сохраняем в: {outputs_dir}")

error_analysis_data = [
    {
        "Модель": "intfloat/multilingual-e5-small",
        "ID вопроса": "q_08",
        "Запрос": "массовая вставка большого количества записей",
        "Ожидалось": "bulkInsertRecords",
        "Категория": "database",
        "Найденные функции": "1. chunkList (utils)\n2. chunk_list (utils)\n3. find_orders_by_user (database)",
        "Причина": "Модель сосредоточилась на признаке «массовость» — большое количество записей. В векторном пространстве этот признак близок к функциям, которые обрабатывают большие объемы данных, таким как chunk, count, format_bytes. Технический аспект «вставить список» утратил значимость на фоне количественного признака."
    },
    {
        "Модель": "paraphrase-multilingual-mpnet-base-v2",
        "ID вопроса": "q_01",
        "Запрос": "как проверить, истёк ли токен?",
        "Ожидалось": "verify_jwt_token",
        "Категория": "auth",
        "Найденные функции": "1. validate_credit_card (validation)\n2. validateCreditCard (validation)\n3. validate_inn (validation)",
        "Причина": "Глагол «проверить» чаще всего встречается в категории валидации в датасете. В векторном пространстве кластер с глаголами 'validate', 'check' и «проверить» очень плотный. Модель не смогла сохранить контекст о том, истёк ли токен. Лексический триггер перебил техническую специфику."
    },
    {
        "Модель": "paraphrase-multilingual-mpnet-base-v2",
        "ID вопроса": "q_24",
        "Запрос": "compute file hash for verification",
        "Ожидалось": "calculateHash",
        "Категория": "utils",
        "Найденные функции": "1. checkPassword (auth)\n2. check_password (auth)\n3. calculate_hash (utils)",
        "Причина": "В датасете функции тесно связаны с триггерами 'check', 'validate'. Запрос включает слово 'verification', которое в этом контексте чаще всего означает проверку данных, а не вычисление хэша. Поэтому безопасность оказалась семантически ближе к этим функциям в векторном пространстве."
    },
    {
        "Модель": "paraphrase-multilingual-MiniLM-L12-v2",
        "ID вопроса": "q_01",
        "Запрос": "как проверить, истёк ли токен?",
        "Ожидалось": "verify_jwt_token",
        "Категория": "auth",
        "Найденные функции": "1. validate_inn (validation)\n2. check_password (auth)\n3. validate_snils (validation)",
        "Причина": "У MiniLM меньшая размерность вектора — 384 против 768. Ей сложнее обрабатывать запросы, включающие несколько концепций (действие, объект и состояние). Модель упростила запросы до наиболее частого паттерна: проверить → validate/check. Из-за ограниченной ёмкости модель потеряла глубину контекста."
    },
    {
        "Модель": "paraphrase-multilingual-MiniLM-L12-v2",
        "ID вопроса": "q_08",
        "Запрос": "массовая вставка большого количества записей",
        "Ожидалось": "bulkInsertRecords",
        "Категория": "database",
        "Найденные функции": "1. countUsers (database)\n2. bulk_insert_records (database)\n3. format_bytes (utils)",
        "Причина": "Модель правильно определила смысл, поставив bulk_insert_records на второе место. Однако эталонный ответ ожидает Java-версию. Оставшиеся две позиции смещены в сторону «количества/размера». Семантика верна, но язык другой, и количественный атрибут повлиял на результат."
    },
    {
        "Модель": "paraphrase-multilingual-MiniLM-L12-v2",
        "ID вопроса": "q_11",
        "Запрос": "обработка ситуации, когда сервер вернул 429",
        "Ожидалось": "handle_rate_limit",
        "Категория": "http",
        "Найденные функции": "1. handleRateLimit (http)\n2. restoreFromBackup (database)\n3. rollbackTransaction (database)",
        "Причина": "Кросс-языковое совпадение: модель обнаружила идентичную Java-функцию на первом месте. Остальные две позиции оказались в категории «восстановление после ошибки», так как в векторном пространстве обработка ошибок сервера близка к rollback/restore."
    },
    {
        "Модель": "paraphrase-multilingual-MiniLM-L12-v2",
        "ID вопроса": "q_24",
        "Запрос": "compute file hash for verification",
        "Ожидалось": "calculateHash",
        "Категория": "utils",
        "Найденные функции": "1. calculate_hash (utils)\n2. validateCredentials (auth)\n3. validateInn (validation)",
        "Причина": "Модель успешно определила правильную логику в функции calculate_hash на Python и поставила ее на первое место. Однако эталонный ответ - JavaScript. Слово verification снова сгруппировало задачи по безопасности и валидации."
    }
]

df = pd.DataFrame(error_analysis_data)

# CSS стили для таблицы (уменьшенные столбцы, кроме причины)
html_code = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Анализ ошибок семантического поиска</title>
    <style>
        body {{
            font-family: 'Segoe UI', Arial, sans-serif;
            margin: 30px;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
        }}
        th {{
            background-color: #34495e;
            color: white;
            font-weight: bold;
            text-align: center;
            padding: 10px 8px;
            border: 1px solid #466;
        }}
        td {{
            border: 1px solid #ddd;
            padding: 8px;
            vertical-align: top;
            text-align: left;
        }}
        td:first-child {{
            text-align: center;
        }}
        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}
        tr:hover {{
            background-color: #f0f0f0;
        }}
        .function-list {{
            white-space: pre-line;
            line-height: 1.5;
        }}
    </style>
</head>
<body>

<h2 style="text-align: center; color: #2c3e50;">Таблица ошибок семантического поиска</h2>

<table>
    <thead>
        <tr>
            <th style="width: 30px;">№</th>
            <th style="width: 200px;">Модель</th>
            <th style="width: 50px;">ID</th>
            <th style="width: 150px;">Запрос</th>
            <th style="width: 100px;">Ожидалось</th>
            <th style="width: 60px;">Категория</th>
            <th style="width: 180px;">Найденные функции (категория)</th>
            <th style="width: 500px;">Причина ошибки</th>
        </tr>
    </thead>
    <tbody>
"""

for i, row in df.iterrows():
    functions_html = row['Найденные функции'].replace('\n', '<br>')
    
    html_code += f"""
        <tr>
            <td>{i+1}</td>
            <td>{row['Модель']}</td>
            <td>{row['ID вопроса']}</td>
            <td>{row['Запрос']}</td>
            <td>{row['Ожидалось']}</td>
            <td>{row['Категория']}</td>
            <td class="function-list">{functions_html}</td>
            <td>{row['Причина']}</td>
        </tr>
    """

html_code += """
    </tbody>
</table>

</body>
</html>
"""

# Отображаем в Jupyter
display(HTML(html_code))

# Сохраняем в outputs 
with open(outputs_dir / "error_analysis_table.html", "w", encoding="utf-8") as f:
    f.write(html_code)
print(f"\nHTML сохранён в {outputs_dir / 'error_analysis_table.html'}")

# Сохраняем в CSV
df.to_csv(outputs_dir / "error_analysis_table.csv", index=False, encoding="utf-8-sig")
print(f"CSV сохранён в {outputs_dir / 'error_analysis_final.csv'}")

print("\n" + "="*60)
print("Файлы сохранены в папку outputs")
print("="*60)

Сохраняем в: C:\CPP\semantic-search-case\outputs


№,Модель,ID,Запрос,Ожидалось,Категория,Найденные функции (категория),Причина ошибки
1,intfloat/multilingual-e5-small,q_08,массовая вставка большого количества записей,bulkInsertRecords,database,1. chunkList (utils)2. chunk_list (utils)3. find_orders_by_user (database),"Модель сосредоточилась на признаке «массовость» — большое количество записей. В векторном пространстве этот признак близок к функциям, которые обрабатывают большие объемы данных, таким как chunk, count, format_bytes. Технический аспект «вставить список» утратил значимость на фоне количественного признака."
2,paraphrase-multilingual-mpnet-base-v2,q_01,"как проверить, истёк ли токен?",verify_jwt_token,auth,1. validate_credit_card (validation)2. validateCreditCard (validation)3. validate_inn (validation),"Глагол «проверить» чаще всего встречается в категории валидации в датасете. В векторном пространстве кластер с глаголами 'validate', 'check' и «проверить» очень плотный. Модель не смогла сохранить контекст о том, истёк ли токен. Лексический триггер перебил техническую специфику."
3,paraphrase-multilingual-mpnet-base-v2,q_24,compute file hash for verification,calculateHash,utils,1. checkPassword (auth)2. check_password (auth)3. calculate_hash (utils),"В датасете функции тесно связаны с триггерами 'check', 'validate'. Запрос включает слово 'verification', которое в этом контексте чаще всего означает проверку данных, а не вычисление хэша. Поэтому безопасность оказалась семантически ближе к этим функциям в векторном пространстве."
4,paraphrase-multilingual-MiniLM-L12-v2,q_01,"как проверить, истёк ли токен?",verify_jwt_token,auth,1. validate_inn (validation)2. check_password (auth)3. validate_snils (validation),"У MiniLM меньшая размерность вектора — 384 против 768. Ей сложнее обрабатывать запросы, включающие несколько концепций (действие, объект и состояние). Модель упростила запросы до наиболее частого паттерна: проверить → validate/check. Из-за ограниченной ёмкости модель потеряла глубину контекста."
5,paraphrase-multilingual-MiniLM-L12-v2,q_08,массовая вставка большого количества записей,bulkInsertRecords,database,1. countUsers (database)2. bulk_insert_records (database)3. format_bytes (utils),"Модель правильно определила смысл, поставив bulk_insert_records на второе место. Однако эталонный ответ ожидает Java-версию. Оставшиеся две позиции смещены в сторону «количества/размера». Семантика верна, но язык другой, и количественный атрибут повлиял на результат."
6,paraphrase-multilingual-MiniLM-L12-v2,q_11,"обработка ситуации, когда сервер вернул 429",handle_rate_limit,http,1. handleRateLimit (http)2. restoreFromBackup (database)3. rollbackTransaction (database),"Кросс-языковое совпадение: модель обнаружила идентичную Java-функцию на первом месте. Остальные две позиции оказались в категории «восстановление после ошибки», так как в векторном пространстве обработка ошибок сервера близка к rollback/restore."
7,paraphrase-multilingual-MiniLM-L12-v2,q_24,compute file hash for verification,calculateHash,utils,1. calculate_hash (utils)2. validateCredentials (auth)3. validateInn (validation),Модель успешно определила правильную логику в функции calculate_hash на Python и поставила ее на первое место. Однако эталонный ответ - JavaScript. Слово verification снова сгруппировало задачи по безопасности и валидации.



HTML сохранён в C:\CPP\semantic-search-case\outputs\error_analysis_table.html
CSV сохранён в C:\CPP\semantic-search-case\outputs\error_analysis_final.csv

Файлы сохранены в папку outputs
